# CLIP per-head attention maps

Where does each attention head of CLIP's vision tower look? This recipe reads the attention
pattern of every head in every layer of CLIP ViT-B/32 and asks which heads send the class token's
attention to the object in the picture. The class token is the position CLIP's image embedding is
read from, so its attention row is the head's answer to "which patches feed the embedding".

Papers: Radford et al., 2021, "Learning Transferable Visual Models From Natural Language
Supervision" (CLIP), https://arxiv.org/abs/2103.00020; Dosovitskiy et al., 2021, "An Image is
Worth 16x16 Words: Transformers for Image Recognition at Scale" (ViT),
https://arxiv.org/abs/2010.11929.

TorchLens gives every `CLIPAttention` module the same per-head facets as a ViT or GPT-2 block:
`q`, `k`, `v`, `scores`, `pattern`, `z` and `result`. transformers runs CLIP's attention through
the fused `scaled_dot_product_attention` kernel, which never materializes the attention matrix;
with `reconstruction_ready=True` TorchLens saves the kernel's actual inputs and rebuilds
`pattern` from them, checked against the kernel's output. The model is the public
`openai/clip-vit-base-patch32` checkpoint (about 600 MB, downloaded once from the Hugging Face
Hub); everything runs on a laptop CPU in about a minute. The pictures are drawn here, so the
object's position is known exactly.

In [ ]:
# recipe-test: tool=clip expected=(4, 3, 224, 224)
import matplotlib.pyplot as plt
import torch
import transformers
from PIL import Image, ImageDraw

import torchlens as tl

MODEL_ID = "openai/clip-vit-base-patch32"
REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"  # pinned so the numbers below reproduce
model = transformers.CLIPModel.from_pretrained(MODEL_ID, revision=REVISION).eval()
processor = transformers.CLIPProcessor.from_pretrained(MODEL_ID, revision=REVISION)

# One disc per picture, in a different corner each time, on a plain background.
SIZE, RADIUS = 224, 40
CORNERS = {"red": (56, 56), "green": (168, 56), "blue": (56, 168), "yellow": (168, 168)}
RGB = {
    "red": (220, 30, 30),
    "green": (30, 170, 50),
    "blue": (30, 60, 220),
    "yellow": (240, 210, 20),
}
pictures, object_masks = [], []
rows, cols = torch.meshgrid(torch.arange(SIZE), torch.arange(SIZE), indexing="ij")
for colour, (x, y) in CORNERS.items():
    picture = Image.new("RGB", (SIZE, SIZE), (245, 245, 245))
    ImageDraw.Draw(picture).ellipse(
        (x - RADIUS, y - RADIUS, x + RADIUS, y + RADIUS), fill=RGB[colour]
    )
    pictures.append(picture)
    disc = (((cols - x) ** 2 + (rows - y) ** 2) <= RADIUS**2).float()
    # ViT-B/32 cuts a 224 x 224 picture into a 7 x 7 grid of 32-pixel patches.
    coverage = disc.reshape(7, 32, 7, 32).mean(dim=(1, 3)).flatten()
    object_masks.append(coverage > 0.25)
object_masks = torch.stack(object_masks)  # [picture, patch]: the patches the disc covers

pixel_values = processor(images=pictures, return_tensors="pt")["pixel_values"]
RESULT = tuple(pixel_values.shape)

## 1. Trace the vision tower and read one head's pattern

`trace.modules[address].facets` is the facet menu of one module. `pattern` is
`[picture, head, query, key]` over the 50 tokens (the class token, then the 49 patches), and
`facets.head(i)` is the same menu for one head.

In [ ]:
# recipe-test: tool=clip expected=((4, 12, 50, 50), (4, 50, 50), True)
trace = tl.trace(
    model.vision_model,
    pixel_values,
    capture=tl.options.CaptureOptions(reconstruction_ready=True),
)
attention = trace.modules["encoder.layers.5.self_attn"]
pattern = attention.facets["pattern"].value
one_head = attention.facets.head(11).pattern
rows_sum_to_one = bool(torch.allclose(pattern.sum(-1), torch.ones(()), atol=1e-5))
print(list(attention.facets.keys()))
RESULT = (tuple(pattern.shape), tuple(one_head.shape), rows_sum_to_one)

The pattern above was rebuilt from the fused kernel's saved inputs. The same model on
transformers' eager attention computes the softmax as an ordinary op, which TorchLens captures
directly; the two agree to float precision.

In [ ]:
# recipe-test: tool=clip expected=True
eager_model = transformers.CLIPModel.from_pretrained(
    MODEL_ID, revision=REVISION, attn_implementation="eager"
).eval()
eager_trace = tl.trace(eager_model.vision_model, pixel_values[:1])
eager_pattern = eager_trace.modules["encoder.layers.5.self_attn"].facets["pattern"].value
difference = (eager_pattern - pattern[:1]).abs().max().item()
print(f"largest difference, rebuilt vs eager pattern: {difference:.1e}")
RESULT = difference < 1e-4

## 2. Which heads look at the object?

For every layer and head, sum the class token's attention over the patches the disc covers and
average over the four pictures. The disc covers 6 of the 49 patches (12 percent), so a head that
ignored the picture's content would put about 0.12 of its patch attention there, and less in
total, since the class token also attends to itself.

In [ ]:
# recipe-test: tool=clip expected=((12, 12), True)
n_layers = len(model.vision_model.encoder.layers)
object_attention = torch.zeros(n_layers, 12)
self_attention = torch.zeros(n_layers)
for layer in range(n_layers):
    facets = trace.modules[f"encoder.layers.{layer}.self_attn"].facets
    cls_row = facets["pattern"].value[:, :, 0, :]  # [picture, head, key]
    on_object = cls_row[:, :, 1:] * object_masks[:, None, :]
    object_attention[layer] = on_object.sum(-1).mean(0)
    self_attention[layer] = cls_row[:, :, 0].mean()

ranked = sorted(
    (
        (object_attention[layer, head].item(), layer, head)
        for layer in range(n_layers)
        for head in range(12)
    ),
    reverse=True,
)
print("object share of the class token's attention, top heads:")
for share, layer, head in ranked[:5]:
    print(f"  layer {layer:2d} head {head:2d}: {share:.2f}")
print("class token attending to itself, mean over heads, by layer:")
print("  " + " ".join(f"{value:.2f}" for value in self_attention.tolist()))
best_share, best_layer, best_head = ranked[0]
RESULT = (tuple(object_attention.shape), best_share > 3 * object_masks.float().mean().item())

The best heads sit in the middle of the tower, and put three to four times the object's area
share on it. Early layers spread the class token's attention evenly, and from the middle layers
on, most of the class token's attention goes to itself.

## 3. Draw the maps

Left: every head's object share, layer by layer. Right: the best head's class-token attention
over the 7 x 7 patch grid, one map per picture.

In [ ]:
# recipe-test: tool=clip expected=5
figure, axes = plt.subplots(1, 5, figsize=(17, 3.4))
image = axes[0].imshow(object_attention, cmap="viridis", aspect="auto")
axes[0].set_xlabel("head")
axes[0].set_ylabel("layer")
axes[0].set_title("class-token attention on the object")
figure.colorbar(image, ax=axes[0])
best_maps = trace.modules[f"encoder.layers.{best_layer}.self_attn"].facets.head(best_head).pattern
for index, (axis, picture) in enumerate(zip(axes[1:], pictures)):
    axis.imshow(picture, extent=(0, 7, 7, 0))
    axis.imshow(best_maps[index, 0, 1:].reshape(7, 7), cmap="magma", alpha=0.6, extent=(0, 7, 7, 0))
    axis.set_title(f"layer {best_layer} head {best_head}")
    axis.axis("off")
figure.tight_layout()
RESULT = len(figure.axes) - 1  # the colour bar is an axes too